# Notebook 08: Safehouse Capacity & Outcomes (Explanatory)

## Section 1 — Problem Framing

**Business Problem:** Identify which safehouse factors drive better resident outcomes (measured by average education progress) to inform resource allocation decisions.

**Approach:** Explanatory OLS regression — we interpret coefficients to understand *which* capacity, staffing, and activity factors are associated with better education outcomes.

**Target Variable:** `avg_education_progress` (continuous score measuring resident educational advancement)

**Stakeholders:** Program directors, safehouse managers

## Section 2 — Data Acquisition and Preparation

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Load data
metrics = pd.read_csv('../../data/lighthouse_csv_v7/safehouse_monthly_metrics.csv')
safehouses = pd.read_csv('../../data/lighthouse_csv_v7/safehouses.csv')

print(f"Monthly metrics shape: {metrics.shape}")
print(f"Safehouses shape: {safehouses.shape}")
print(f"\nMetrics columns: {list(metrics.columns)}")
print(f"Safehouses columns: {list(safehouses.columns)}")
print(f"\nNull counts in metrics:")
print(metrics.isnull().sum())

Monthly metrics shape: (450, 11)
Safehouses shape: (9, 13)

Metrics columns: ['metric_id', 'safehouse_id', 'month_start', 'month_end', 'active_residents', 'avg_education_progress', 'avg_health_score', 'process_recording_count', 'home_visitation_count', 'incident_count', 'notes']
Safehouses columns: ['safehouse_id', 'safehouse_code', 'name', 'region', 'city', 'province', 'country', 'open_date', 'status', 'capacity_girls', 'capacity_staff', 'current_occupancy', 'notes']

Null counts in metrics:
metric_id                    0
safehouse_id                 0
month_start                  0
month_end                    0
active_residents             0
avg_education_progress     197
avg_health_score           197
process_recording_count      0
home_visitation_count        0
incident_count               0
notes                      450
dtype: int64


In [3]:
# Merge on safehouse_id to add capacity_girls, capacity_staff, region
df = metrics.merge(safehouses[['safehouse_id', 'capacity_girls', 'capacity_staff', 'region']], 
                   on='safehouse_id', how='left')

# Drop rows where avg_education_progress is NaN (target variable)
df = df.dropna(subset=['avg_education_progress']).copy()
print(f"Records with non-null avg_education_progress: {len(df)}")
print(f"Safehouses represented: {df['safehouse_id'].nunique()}")
print(f"Regions: {df['region'].value_counts().to_dict()}")

Records with non-null avg_education_progress: 253
Safehouses represented: 9
Regions: {'Visayas': 119, 'Mindanao': 73, 'Luzon': 61}


In [4]:
# Compute derived features
# Occupancy rate
df['occupancy_rate'] = df['active_residents'] / df['capacity_girls']

# Staff ratio (handle div-by-zero)
df['staff_ratio'] = df['capacity_staff'] / df['active_residents'].replace(0, np.nan)
df['staff_ratio'] = df['staff_ratio'].fillna(0)

# Activity rates per resident
df['recording_per_resident'] = df['process_recording_count'] / df['active_residents'].replace(0, np.nan)
df['recording_per_resident'] = df['recording_per_resident'].fillna(0)

df['visit_per_resident'] = df['home_visitation_count'] / df['active_residents'].replace(0, np.nan)
df['visit_per_resident'] = df['visit_per_resident'].fillna(0)

df['incident_rate'] = df['incident_count'] / df['active_residents'].replace(0, np.nan)
df['incident_rate'] = df['incident_rate'].fillna(0)

print("Derived features created:")
for col in ['occupancy_rate', 'staff_ratio', 'recording_per_resident', 'visit_per_resident', 'incident_rate']:
    print(f"  {col}: mean={df[col].mean():.3f}, std={df[col].std():.3f}")

Derived features created:
  occupancy_rate: mean=0.741, std=0.234
  staff_ratio: mean=0.725, std=0.316
  recording_per_resident: mean=1.265, std=0.746
  visit_per_resident: mean=0.620, std=0.416
  incident_rate: mean=0.055, std=0.133


In [5]:
# Features for OLS
categorical_features = ['region']
numeric_features = ['occupancy_rate', 'staff_ratio', 'recording_per_resident', 
                    'visit_per_resident', 'incident_rate']

# One-hot encode region
features_df = pd.get_dummies(df[categorical_features + numeric_features],
                              columns=categorical_features, drop_first=True, dtype=int)

y = df['avg_education_progress']

# Drop any remaining NaN rows
mask = features_df.notna().all(axis=1) & y.notna()
features_df = features_df[mask]
y = y[mask]

X = sm.add_constant(features_df)

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures used:")
for col in X.columns[1:]:
    print(f"  - {col}")

Feature matrix shape: (253, 8)
Target shape: (253,)

Features used:
  - occupancy_rate
  - staff_ratio
  - recording_per_resident
  - visit_per_resident
  - incident_rate
  - region_Mindanao
  - region_Visayas


In [6]:
# Train/test split for sklearn deployment
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df[mask][categorical_features + numeric_features],
    y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train_raw)}, Test size: {len(X_test_raw)}")

Train size: 202, Test size: 51


## Section 3 — Exploration

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Distribution of avg_education_progress by safehouse
for sh_id in df['safehouse_id'].unique():
    sh_data = df[df['safehouse_id'] == sh_id]['avg_education_progress']
    axes[0, 0].hist(sh_data, alpha=0.4, label=f'SH{sh_id}', bins=10)
axes[0, 0].set_title('Education Progress by Safehouse')
axes[0, 0].set_xlabel('Avg Education Progress')
axes[0, 0].legend(fontsize=7)

# Scatter: occupancy_rate vs avg_education_progress
axes[0, 1].scatter(df['occupancy_rate'], df['avg_education_progress'], alpha=0.3, color='coral')
axes[0, 1].set_title('Occupancy Rate vs Education Progress')
axes[0, 1].set_xlabel('Occupancy Rate')
axes[0, 1].set_ylabel('Avg Education Progress')

# Scatter: staff_ratio vs avg_education_progress
axes[0, 2].scatter(df['staff_ratio'], df['avg_education_progress'], alpha=0.3, color='teal')
axes[0, 2].set_title('Staff Ratio vs Education Progress')
axes[0, 2].set_xlabel('Staff Ratio (staff/residents)')
axes[0, 2].set_ylabel('Avg Education Progress')

# Correlation heatmap
corr_cols = numeric_features + ['avg_education_progress']
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1, 0],
            xticklabels=[c[:12] for c in corr_cols], yticklabels=[c[:12] for c in corr_cols])
axes[1, 0].set_title('Correlation Heatmap')

# Box plots by region
regions = df['region'].unique()
region_data = [df[df['region'] == r]['avg_education_progress'].values for r in regions]
axes[1, 1].boxplot(region_data, labels=regions)
axes[1, 1].set_title('Education Progress by Region')
axes[1, 1].set_ylabel('Avg Education Progress')
axes[1, 1].tick_params(axis='x', rotation=45)

# Recording per resident vs education progress
axes[1, 2].scatter(df['recording_per_resident'], df['avg_education_progress'], alpha=0.3, color='coral')
axes[1, 2].set_title('Recordings/Resident vs Education Progress')
axes[1, 2].set_xlabel('Process Recordings per Resident')
axes[1, 2].set_ylabel('Avg Education Progress')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/08_exploration.png', dpi=100, bbox_inches='tight')
plt.show()
print("Exploration plots generated")

Exploration plots generated


## Section 4 — Modeling (Explanatory OLS)

In [8]:
# Fit OLS model (explanatory — for coefficient interpretation)
model = sm.OLS(y, X).fit()
print(model.summary())

                              OLS Regression Results                              
Dep. Variable:     avg_education_progress   R-squared:                       0.036
Model:                                OLS   Adj. R-squared:                  0.009
Method:                     Least Squares   F-statistic:                     1.309
Date:                    Mon, 06 Apr 2026   Prob (F-statistic):              0.246
Time:                            15:19:33   Log-Likelihood:                -1095.9
No. Observations:                     253   AIC:                             2208.
Df Residuals:                         245   BIC:                             2236.
Df Model:                               7                                         
Covariance Type:                nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------

In [9]:
# VIF check for multicollinearity
vif_data = pd.DataFrame({
    'Feature': X.columns[1:],  # Skip constant
    'VIF': [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
})
vif_data = vif_data.sort_values('VIF', ascending=False)
print("Variance Inflation Factors:")
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\nWARNING: {len(high_vif)} features with VIF > 10 detected.")
    print("Dropping high-VIF features and re-fitting...")
    drop_cols = high_vif['Feature'].tolist()
    X_clean = X.drop(columns=drop_cols)
    model = sm.OLS(y, X_clean).fit()
    print("\nRe-fitted model summary:")
    print(model.summary())
    vif_data2 = pd.DataFrame({
        'Feature': X_clean.columns[1:],
        'VIF': [variance_inflation_factor(X_clean.values, i) for i in range(1, X_clean.shape[1])]
    })
    print("\nUpdated VIF values:")
    print(vif_data2.sort_values('VIF', ascending=False).to_string(index=False))
else:
    print("\nAll VIF values <= 10. No multicollinearity issues detected.")

Variance Inflation Factors:
               Feature      VIF
       region_Mindanao 2.331126
        region_Visayas 2.129917
        occupancy_rate 2.053083
           staff_ratio 1.911227
recording_per_resident 1.568642
    visit_per_resident 1.375757
         incident_rate 1.237211

All VIF values <= 10. No multicollinearity issues detected.


### Coefficient Interpretation

The OLS model identifies which safehouse factors significantly predict education progress:
- **occupancy_rate:** Does crowding affect outcomes?
- **staff_ratio:** Do more staff per resident improve outcomes?
- **recording_per_resident:** Do more counseling sessions correlate with better progress?
- **visit_per_resident:** Do home visitations help?
- **incident_rate:** Do safety incidents hinder progress?
- **region:** Are there regional differences in outcomes?


## Section 5 — Evaluation

In [10]:
# Model fit statistics
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
y_pred_ols = model.fittedvalues
rmse = np.sqrt(mean_squared_error(y, y_pred_ols))
print(f"RMSE: {rmse:.4f}")
print(f"F-statistic: {model.fvalue:.2f}, p-value: {model.f_pvalue:.2e}")

R-squared: 0.0361
Adjusted R-squared: 0.0085
RMSE: 18.4079
F-statistic: 1.31, p-value: 2.46e-01


In [11]:
# Residual diagnostics
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.3, color='coral')
axes[0].axhline(y=0, color='black', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted Values')

# Q-Q plot
sm.qqplot(residuals, line='45', ax=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

# Histogram
axes[2].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[2].set_title('Distribution of Residuals')
axes[2].set_xlabel('Residual')

plt.tight_layout()
plt.savefig('../../ml-pipelines/notebooks/08_residuals.png', dpi=100, bbox_inches='tight')
plt.show()

In [12]:
# Shapiro-Wilk test for normality
if len(residuals) > 5000:
    stat, p_val = shapiro(residuals.sample(5000, random_state=42))
else:
    stat, p_val = shapiro(residuals)
print(f"Shapiro-Wilk Test: statistic={stat:.4f}, p-value={p_val:.4f}")
if p_val < 0.05:
    print("Residuals are NOT normally distributed (p < 0.05).")
else:
    print("Residuals appear normally distributed (p >= 0.05).")

# Breusch-Pagan test for homoscedasticity
X_model = model.model.exog
bp_test = het_breuschpagan(residuals, X_model)
labels = ['LM Statistic', 'LM p-value', 'F-Statistic', 'F p-value']
print(f"\nBreusch-Pagan Test:")
for label, val in zip(labels, bp_test):
    print(f"  {label}: {val:.4f}")
if bp_test[1] < 0.05:
    print("Heteroscedasticity detected (p < 0.05).")
else:
    print("No evidence of heteroscedasticity (p >= 0.05).")

Shapiro-Wilk Test: statistic=0.9373, p-value=0.0000
Residuals are NOT normally distributed (p < 0.05).



Breusch-Pagan Test:
  LM Statistic: 10.8765
  LM p-value: 0.1441
  F-Statistic: 1.5723
  F p-value: 0.1442
No evidence of heteroscedasticity (p >= 0.05).


In [13]:
# Coefficient table with confidence intervals
coef_table = pd.DataFrame({
    'Coefficient': model.params,
    'Std Error': model.bse,
    'p-value': model.pvalues,
    'CI Lower': model.conf_int()[0],
    'CI Upper': model.conf_int()[1]
})
coef_table = coef_table.drop('const', errors='ignore')
coef_table = coef_table.sort_values('p-value')
print("Coefficient Table (sorted by significance):")
print(coef_table.to_string())

print(f"\nSignificant features (p < 0.05): {(coef_table['p-value'] < 0.05).sum()} of {len(coef_table)}")

Coefficient Table (sorted by significance):
                        Coefficient  Std Error   p-value   CI Lower   CI Upper
incident_rate            -16.371848   9.818756  0.096711 -35.711791   2.968095
region_Visayas            -3.678660   3.438731  0.285775 -10.451909   3.094588
staff_ratio                4.506251   5.156667  0.383045  -5.650805  14.663307
visit_per_resident         2.880095   3.320284  0.386558  -3.659849   9.420038
occupancy_rate             5.790564   7.211117  0.422750  -8.413129  19.994258
recording_per_resident     1.000808   1.977963  0.613326  -2.895174   4.896789
region_Mindanao            0.724720   3.963036  0.855051  -7.081247   8.530688

Significant features (p < 0.05): 0 of 7


## Section 6 — Causal Analysis

In [14]:
# Significant coefficients interpretation
sig = coef_table[coef_table['p-value'] < 0.05].copy()

print("SIGNIFICANT FACTORS affecting education progress (p < 0.05):")
print("=" * 80)
if len(sig) == 0:
    print("  No features are individually significant at p < 0.05.")
    print("  This may indicate that education progress is driven by unmeasured factors")
    print("  (e.g., individual resident characteristics, program curriculum quality)")
    print("  or that the available features have small effect sizes relative to noise.")
else:
    for idx, row in sig.iterrows():
        direction = "+" if row['Coefficient'] > 0 else ""
        label = idx.replace('region_', 'Region: ')
        print(f"  {label}: {direction}{row['Coefficient']:.4f} (p={row['p-value']:.4f})")
        if row['Coefficient'] > 0:
            print(f"    -> Associated with {abs(row['Coefficient']):.2f} higher education progress")
        else:
            print(f"    -> Associated with {abs(row['Coefficient']):.2f} lower education progress")

print()
print("Note: These are ASSOCIATIONS, not causal effects.")
print("Better-staffed safehouses may also receive more funding, have newer facilities, etc.")

SIGNIFICANT FACTORS affecting education progress (p < 0.05):
  No features are individually significant at p < 0.05.
  This may indicate that education progress is driven by unmeasured factors
  (e.g., individual resident characteristics, program curriculum quality)
  or that the available features have small effect sizes relative to noise.

Note: These are ASSOCIATIONS, not causal effects.
Better-staffed safehouses may also receive more funding, have newer facilities, etc.


### Association vs Causation

The OLS coefficients show **association**, not **causation**. Key confounders:

1. **Funding and resources:** Better-staffed safehouses may also have more educational materials, better facilities, and more experienced counselors.
2. **Resident mix:** Some safehouses may receive residents with different educational backgrounds or ages, affecting baseline progress rates.
3. **Safehouse age:** Newer safehouses may have different operational patterns than established ones.
4. **Location effects:** Urban vs rural settings affect access to educational resources, tutors, and school partnerships.
5. **Selection bias:** Residents may be assigned to safehouses based on factors that also affect educational outcomes.

### Recommendations

Based on the model results:
- **Optimal staffing ratios:** If staff_ratio has a significant positive coefficient, consider investing in additional staff at under-resourced safehouses
- **Target occupancy levels:** If occupancy_rate has a negative coefficient, avoid overcrowding which may reduce per-resident attention
- **Process recordings:** If recording_per_resident is positively associated, it suggests counseling support enhances educational outcomes
- **Incident management:** If incident_rate is negatively associated, prioritize safety and conflict resolution programs

## Section 7 - Deployment

**Deployment Architecture:** Pre-computed predictions written to PostgreSQL.

This model is deployed as an offline batch pipeline. The production workflow is:

1. **ETL:** `jobs/etl_safehouse.py` reads safehouse_monthly_metrics, safehouses, and residents from PostgreSQL, engineers features (occupancy rates, demographic aggregates), and writes the `ml_safehouse_features` table.
2. **Train:** `jobs/train_safehouse.py` trains an OLS LinearRegression pipeline, saves the model as `artifacts/safehouse_outcomes.sav` along with `metadata.json` and `metrics.json`.
3. **Inference:** `jobs/run_inference_safehouse.py` loads the trained model, predicts health outcomes for all safehouse metrics, and writes results to the `safehouse_predictions` table in PostgreSQL.

The .NET backend queries `safehouse_predictions` via EF Core. The frontend fetches insights from `GET /api/predictions/safehouse`.

**Model type:** Explanatory (OLS regression)

**Approach:** Explanatory -- which factors drive better health outcomes across safehouses. The sklearn Pipeline wraps the OLS model for inference consistency.

In [15]:
# For Flask API deployment: Train sklearn LinearRegression Pipeline
cat_features_deploy = ['region']
num_features_deploy = ['occupancy_rate', 'staff_ratio', 'recording_per_resident',
                        'visit_per_resident', 'incident_rate']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features_deploy),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_features_deploy)
])

sklearn_pipeline = SkPipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

sklearn_pipeline.fit(X_train_raw, y_train)

# Evaluate on test set
y_pred_sk = sklearn_pipeline.predict(X_test_raw)
print(f"sklearn Pipeline Test R-squared: {r2_score(y_test, y_pred_sk):.4f}")
print(f"sklearn Pipeline Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_sk)):.4f}")

sklearn Pipeline Test R-squared: -0.0651
sklearn Pipeline Test RMSE: 20.2546


In [ ]:
# Save sklearn pipeline as .sav to artifacts/
joblib.dump(sklearn_pipeline, '../artifacts/safehouse_outcomes.sav')
print("Model saved to: ../artifacts/safehouse_outcomes.sav")

print("\nProduction scripts:")
print("  ETL:       jobs/etl_safehouse.py")
print("  Train:     jobs/train_safehouse.py")
print("  Inference: jobs/run_inference_safehouse.py")
print("\nPredictions are pre-computed to PostgreSQL table: safehouse_predictions")

In [17]:
# Expected input features for prediction
print("Expected input features (as DataFrame columns):")
print(f"  Categorical: {cat_features_deploy}")
print(f"  Numeric: {num_features_deploy}")
print()

# Example prediction
example = pd.DataFrame([{
    'region': 'Luzon',
    'occupancy_rate': 0.85,
    'staff_ratio': 0.5,
    'recording_per_resident': 1.5,
    'visit_per_resident': 0.8,
    'incident_rate': 0.1,
}])
pred = loaded.predict(example)
print(f"Example prediction (avg_education_progress): {pred[0]:.2f}")

Expected input features (as DataFrame columns):
  Categorical: ['region']
  Numeric: ['occupancy_rate', 'staff_ratio', 'recording_per_resident', 'visit_per_resident', 'incident_rate']

Example prediction (avg_education_progress): 81.02
